<a href="https://colab.research.google.com/github/alex-jk/YRP-vehicle-accidents/blob/main/yrp_vehicle_accidents_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import YRP data

In [9]:
from getpass import getpass
TOKEN = getpass('GitHub token: ')

!git clone https://{TOKEN}@github.com/alex-jk/YRP-vehicle-accidents.git
%cd YRP-vehicle-accidents

GitHub token: ··········
Cloning into 'YRP-vehicle-accidents'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 31 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 18.70 KiB | 9.35 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/YRP-vehicle-accidents/YRP-vehicle-accidents/YRP-vehicle-accidents/YRP-vehicle-accidents


In [10]:
import pandas as pd
import re
df = pd.read_csv("data/YRP Data vehicle accidents 2023 - now.csv")

# Parse year
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year

# People per row: count items in Age/Gender (comma- or slash-separated); use max; default to 1
def count_items(x):
    if not isinstance(x, str): return 0
    parts = [p.strip() for p in re.split(r'[,/]', x) if p.strip()]
    return len(parts)

age_n    = df['Age'].apply(count_items)
gender_n = df['Gender'].apply(count_items)
person_n = age_n.combine(gender_n, max).where(lambda s: s>0, 1)
# print(person_n)

# Robust Y/N → boolean
is_yes = lambda x: isinstance(x, str) and x.strip().upper().startswith('Y')
mot = df['Motorcycle'].map(is_yes)
ped = df['Pedestrian'].map(is_yes)
cyc = df['Cyclist'].map(is_yes)
mob = df['Mobility Scooter'].map(is_yes)

# Category per row (priority: Pedestrian > Cyclist > Mobility scooter > Other)
def victim_type(mot, ped, cyc, mob):
    if mot: return 'Motorcyclist'
    if ped: return 'Pedestrian'
    if cyc: return 'Cyclist'
    if mob: return 'Mobility Scooter'
    return 'Car'

df['Type'] = [victim_type(m, p, c, b) for m, p, c, b in zip(mot, ped, cyc, mob)]
df['Deaths'] = person_n

print(f"\nDF shape: {df.shape}")
df.head(10)


DF shape: (68, 18)


,ID,Date,Time,Main Street,Main Street 2,Municipality,Intersection,Motorcycle,Age,Gender,Mobility Scooter,Pedestrian,Single Vehicle,Night,Cyclist,Year,Type,Deaths
0,2023_16704,2023-01-14,14:28,McCowan Road,Bur Oak Avenue,Markham,Y,N,75,Female,N,N,N,N,N,2023,Car,1
1,2023_35339,2023-01-30,6:30,9th Line,Bloomington Road,Whitchurch-Stouffville,Y,N,"Adult, Adult","Male, Male",N,N,N,N,N,2023,Car,2
2,2023_87652,2023-03-15,12:49,Highway 48,NaN,Whitchurch-Stouffville,N,N,35,Male,N,N,N,N,N,2023,Car,1
3,2023_170962,2023-05-19,23:49,Major MacKenzie Dr W,Jane Street,Vaughan,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,Car,1
4,2023_175551,2023-05-23,18:11,Pine Valley Drive,Major MacKenzie Dr W,Vaughan,Y,N,20,Male,N,Y,N,N,N,2023,Pedestrian,1
5,2023_183025,2023-05-29,7:52,Jane Street,NaN,King,Y,Y,Adult,Male,N,N,N,N,N,2023,Motorcyclist,1
6,2023_189685,2023-06-01,19:21,Pine Valley Drive,NaN,Vaughan,N,N,72,Female,N,N,N,N,N,2023,Car,1
7,2023_238781,2023-07-05,12:22,Dudley Avenue,NaN,Markham,N,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,Car,1
8,2023_265277,2023-07-25,22:15,Highway 7,Thornhill Woods Drive,Vaughan,Y,Y,"45, 47","Male, Female",N,N,N,N,N,2023,Motorcyclist,2
9,2023_269592,2023-07-29,10:23,Keele Street,Sherwood Park Dr,Vaughan,Y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2023,Car,1


Summarize by year and type

In [14]:
# Sum people by year & type
counts = (
    df.groupby(['Year','Type'])['Deaths']
      .sum()
      .unstack('Type')
      .fillna(0)
      .astype(int)
      .sort_index()
)

# totals (people) per year
year_totals = df.groupby('Year')['Deaths'].sum()

def gender_counts(cell, deaths):
    # NA/blank → all Unknown
    if not isinstance(cell, str) or not cell.strip():
        return pd.Series({'Male Total': 0, 'Female Total': 0, 'Unknown Total': deaths})
    toks = [t.strip().lower() for t in re.split(r'[,/;|]', cell) if t.strip()]
    m = sum(t.startswith('m') for t in toks)        # M / Male
    f = sum(t.startswith('f') for t in toks)        # F / Female
    other = sum(not (t.startswith('m') or t.startswith('f')) for t in toks)
    u = other + max(0, deaths - (m + f + other))    # any extra people → Unknown
    return pd.Series({'Male Total': m, 'Female Total': f, 'Unknown Total': u})

# row → year totals
totals_cols = ['Male Total','Female Total','Unknown Total']
by_year_gender = (
    pd.concat([df['Year'],
               df.apply(lambda r: gender_counts(r['Gender'], int(r['Deaths'])), axis=1)], axis=1)
      .groupby('Year')[totals_cols].sum()
)

# Make the table tidy, ordered, with totals
nice = counts.copy()

# attach to counts table
for col in ['Male Total','Female Total','Unknown Total']:
    nice[col] = by_year_gender.reindex(nice.index.drop('All years', errors='ignore')).get(col)

# "All years" totals
nice.loc['All years', totals_cols] = by_year_gender.sum().values

# ensure all expected columns exist
select_cols = ['Motorcyclist', 'Pedestrian', 'Cyclist', 'Mobility Scooter', 'Car']
all_cols = select_cols + totals_cols
for col in all_cols:
    if col not in nice.columns:
        nice[col] = 0

# order columns and add totals
nice = nice[all_cols]
nice['Total'] = nice[select_cols].sum(axis=1)
nice.loc['All years'] = nice.sum()

# ints + a simple style (optional in notebooks)
nice = nice.astype(int)

# pct columns per row
den = (nice[totals_cols].sum(axis=1)).replace(0, pd.NA)
nice['Male %']    = (nice['Male Total']    / den * 100).round(1)
nice['Female %']  = (nice['Female Total']  / den * 100).round(1)
nice['Unknown %'] = (100 - nice['Male %'] - nice['Female %']).round(1)

nice.style.set_caption("Fatalities by Year and Victim Type") \
          .set_table_styles([{'selector': 'caption', 'props': [('font-weight', 'bold')]}]) \
          .format('{:,}')

Type,Motorcyclist,Pedestrian,Cyclist,Mobility Scooter,Car,Male Total,Female Total,Unknown Total,Total,Male %,Female %,Unknown %
Year,,,,,,,,,,,,
2023,5,5,0,0,13,11,7,5,23,47.8,30.4,21.8
2024,5,7,1,1,14,22,5,1,28,78.6,17.9,3.5
2025,3,5,1,0,10,11,4,4,19,57.9,21.1,21.0
All years,13,17,2,1,37,88,32,20,70,62.9,22.9,14.2


Summary by gender and type

In [15]:
# By TYPE × GENDER with a clean display (uses your existing: df, gender_counts, totals_cols, select_cols)

# totals by Type (no hard-coded names)
by_type_gender = (
    pd.concat([df['Type'],
               df.apply(lambda r: gender_counts(r['Gender'], int(r['Deaths'])), axis=1)], axis=1)
      .groupby('Type')[totals_cols].sum()
).reindex(select_cols, fill_value=0)

# add totals and percents
by_type_gender['Total'] = by_type_gender[totals_cols].sum(axis=1)
percent_cols = [c.replace('Total', '%') for c in totals_cols]
for tcol, pcol in zip(totals_cols, percent_cols):
    by_type_gender[pcol] = (by_type_gender[tcol] / by_type_gender['Total'] * 100).round(1)

# tidy order
ordered_cols = totals_cols + ['Total'] + percent_cols
tbl_disp = by_type_gender[ordered_cols].copy()
tbl_disp[totals_cols + ['Total']] = tbl_disp[totals_cols + ['Total']].astype(int)

# nice display
tbl_disp.style.set_caption("By Type — Gender Totals and %") \
    .set_table_styles([{'selector': 'caption', 'props': [('font-weight','bold')]}]) \
    .format({**{c: '{:,}' for c in totals_cols + ['Total']},
             **{c: '{:.1f}%' for c in percent_cols}})

,Male Total,Female Total,Unknown Total,Total,Male %,Female %,Unknown %
Type,,,,,,,
Motorcyclist,11,2,0,13,84.6%,15.4%,0.0%
Pedestrian,11,6,0,17,64.7%,35.3%,0.0%
Cyclist,2,0,0,2,100.0%,0.0%,0.0%
Mobility Scooter,1,0,0,1,100.0%,0.0%,0.0%
Car,19,8,10,37,51.4%,21.6%,27.0%


Summary by age group and type

In [17]:
def age_bucket_counts(cell, deaths):
    out = {'Under 20': 0, 'Adult': 0, '60+': 0, 'Age Unknown': 0}
    if not isinstance(cell, str) or not cell.strip():
        out['Age Unknown'] = deaths; return pd.Series(out)
    toks = [t.strip() for t in re.split(r'[,/;|]', cell) if t.strip()]
    for t in toks:
        tl = t.lower()
        if tl == 'adult': out['Adult'] += 1
        elif tl == 'youth': out['Under 20'] += 1
        elif tl in {'na','n/a','unknown'}: out['Age Unknown'] += 1
        elif tl == '60+': out['60+'] += 1
        else:
            digits = re.sub(r'[^\d]', '', tl)
            if digits:
                n = int(digits)
                out['60+' if n>=60 else 'Adult' if n>=20 else 'Under 20'] += 1
            else:
                out['Age Unknown'] += 1
    missing = max(0, deaths - sum(out.values()))
    if missing: out['Age Unknown'] += missing
    return pd.Series(out)

age_cols = ['Under 20', 'Adult', '60+', 'Age Unknown']

# aggregate by Type
by_type_age = (
    pd.concat([df['Type'],
               df.apply(lambda r: age_bucket_counts(r['Age'], int(r['Deaths'])), axis=1)], axis=1)
      .groupby('Type')[age_cols].sum()
).reindex(select_cols, fill_value=0)

# counts + percents
by_type_age['Total'] = by_type_age[age_cols].sum(axis=1)
pct_cols = [f"{c} %" for c in age_cols]
by_type_age[pct_cols] = (by_type_age[age_cols].div(by_type_age['Total'], axis=0)*100).round(1)

# ---- bottom totals row ----
tot_counts = by_type_age[age_cols].sum()
tot_total  = int(tot_counts.sum())
tot_pcts   = (tot_counts / tot_total * 100).round(1)
by_type_age.loc['All types', age_cols] = tot_counts.values
by_type_age.loc['All types', 'Total']  = tot_total
by_type_age.loc['All types', pct_cols] = tot_pcts.values

# tidy display
ordered_cols = age_cols + ['Total'] + pct_cols
tbl = by_type_age[ordered_cols].copy()
tbl[age_cols + ['Total']] = tbl[age_cols + ['Total']].astype(int)

tbl.style.set_caption("By Type — Age Buckets (counts and %)") \
    .set_table_styles([{'selector':'caption','props':[('font-weight','bold')]}]) \
    .format({**{c:'{:,}' for c in age_cols+['Total']}, **{c:'{:.1f}%' for c in pct_cols}})

,Under 20,Adult,60+,Age Unknown,Total,Under 20 %,Adult %,60+ %,Age Unknown %
Type,,,,,,,,,
Motorcyclist,2,11,0,0,13,15.4%,84.6%,0.0%,0.0%
Pedestrian,4,5,8,0,17,23.5%,29.4%,47.1%,0.0%
Cyclist,1,1,0,0,2,50.0%,50.0%,0.0%,0.0%
Mobility Scooter,0,0,1,0,1,0.0%,0.0%,100.0%,0.0%
Car,4,15,10,8,37,10.8%,40.5%,27.0%,21.6%
All types,11,32,19,8,70,15.7%,45.7%,27.1%,11.4%
